# UAV LiDAR → DTM / DSM / CHM Workflow
**Ground classification via CSF (Zhang et al. 2016) and TIN interpolation**
**Chunked & parallelized for large point clouds**

## Overview

This workflow processes a UAV LiDAR `.las` file through the following steps:

1.  Load and inspect the point cloud
2.  Classify ground points using the **Cloth Simulation Filter (CSF)**
3.  Generate a **Digital Terrain Model (DTM)** via TIN interpolation
4.  Height-normalize the point cloud
5.  Generate a **Digital Surface Model (DSM)** from first returns
6.  Subtract to produce a **Canopy Height Model (CHM)**
7.  Clip to the boundary of the Area of Interest
8.  Summarize canopy height statistics

---

## Setup

Install required packages if you haven't already:

In [ ]:
## install.packages(c("terra", "RCSF", "sf", "lidR", "future"))

In [ ]:
library(terra)
library(sf)
library(RCSF)
library(lidR)
library(future)

# Reserve one core for OS tasks (file I/O, memory management, etc.)
plan(multisession, workers = parallel::detectCores() - 1)

out_dir <- readline(prompt = "Enter output folder path: ")
cat("Saving outputs to:", out_dir, "\n")

# Create subdirectories for intermediate tiled outputs
dir.create(file.path(out_dir, "classified"),  showWarnings = FALSE)
dir.create(file.path(out_dir, "normalized"),  showWarnings = FALSE)

---

## 1. Load the Point Cloud

`file.choose()` opens a native OS file browser so you can navigate to
your `.las` or `.laz` file interactively, rather than hardcoding a path.

The file is loaded as a **catalog** rather than a single LAS object.
This tells lidR to tile and parallelize all subsequent steps rather than
processing the full file in one single-threaded pass.

> **Note:** the dialog may open behind your RStudio/VS Code window — check your taskbar.

In [ ]:
message("Please select your LAS or LAZ file in the file browser.")

las_path <- file.choose()
cat("Selected file:", las_path, "\n")

# Load as a catalog so lidR can tile and parallelize all subsequent steps
ctg <- readLAScatalog(las_path,
                      select = "xyzrci",           # load x, y, z, return number, classification, intensity
                      filter = "-drop_z_below 0")  # drop obviously bad points below ground

# Quick check: point count, CRS, tile layout
print(ctg)

---

## 2. Ground Classification with CSF

CSF (Zhang et al. 2016) simulates a cloth dropped over an **inverted**
point cloud. The cloth drapes down under gravity and the points that end
up close to the cloth are labelled as ground.

Key parameters:

| Parameter | What it controls |
|---|---|
| `sloop_smooth` | Post-processing step to recover ground points on steep slopes. `FALSE` for flat terrain. |
| `class_threshold` | Max distance (m) a point can be from the cloth to still be called ground |
| `cloth_resolution` | Grid cell size of the simulated cloth (m) — smaller = finer detail, slower |
| `rigidness` | How much the cloth resists bending: 1 = flexible (hilly), 2 = medium, 3 = rigid (flat) |
| `iterations` | More iterations = cloth drapes more accurately, but slower |
| `time_step` | Internal physics timestep — leave at default 0.65 |

In [ ]:
# Each tile is 500 x 500 m — reduce to 250 if you run into memory pressure
opt_chunk_size(ctg)      <- 500

# 30 m overlap between tiles so the cloth has enough neighboring points
# at tile edges to classify correctly — prevents seams in the output
opt_chunk_buffer(ctg)    <- 30

# Write each classified tile to disk named by its spatial position
opt_output_files(ctg)    <- file.path(out_dir, "classified/tile_{XLEFT}_{YBOTTOM}")

# Compress output to .laz to save disk space (lossless, lidR reads it identically to .las)
opt_laz_compression(ctg) <- TRUE

ctg_classified <- classify_ground(
  ctg,
  algorithm = csf(
    sloop_smooth     = FALSE,  # FALSE for flat/gentle terrain
    class_threshold  = 0.5,    # points within 0.5 m of cloth are classified as ground
    cloth_resolution = 0.5,    # cloth cell size in metres
    rigidness        = 2,      # 2 = medium; use 3 for very flat/open terrain
    iterations       = 500,
    time_step        = 0.65
  )
)

cat("Ground classification complete\n")

---

## 3. DTM via Adaptive Triangulation

`rasterize_terrain()` converts the ground-classified points into a
continuous raster elevation surface — the DTM.

**How TIN interpolation works:**

1.  Takes all ground-classified points
2.  Connects them into a mesh of non-overlapping triangles (Delaunay triangulation)
3.  Triangle sizes naturally adapt to local point density — smaller where points are dense, larger where they are sparse
4.  For each output raster cell, elevation is interpolated from the plane defined by the three vertices of whichever triangle contains that cell centre

This approach honours every ground point exactly without smoothing,
making it the most accurate method for DTM generation from lidar (Mishra & Singh 2026).

In [ ]:
# Clear output files so rasterize_terrain() returns a single merged raster
# rather than writing one raster per tile
opt_output_files(ctg_classified) <- ""

dtm <- rasterize_terrain(
  ctg_classified,
  res       = 0.25,   # 0.25 m pixel size (following Fu et al. 2025)
  algorithm = tin()   # Delaunay TIN built from ground-classified points
)

# Alternative: IDW — faster but slightly smoother/less faithful to ground points
# dtm <- rasterize_terrain(ctg_classified, res = 0.25, algorithm = knnidw(k = 10, p = 2))

writeRaster(dtm, file.path(out_dir, "dtm_0.25m.tif"), overwrite = TRUE)
cat("DTM written to dtm_0.25m.tif\n")

---

## 4. Height-Normalize the Point Cloud

Subtracts the terrain elevation (via TIN) from every point's Z value, so that ground
points become Z ≈ 0 and all other points represent **height above
ground** rather than height above sea level.

Each normalized tile is written to disk and later read back as a catalog
for step 5 — avoids holding the entire normalized cloud in memory.

In [ ]:
# Write each normalized tile to disk named by its spatial position
opt_output_files(ctg_classified)    <- file.path(out_dir, "normalized/tile_{XLEFT}_{YBOTTOM}")
opt_laz_compression(ctg_classified) <- TRUE

# Run normalization — subtracts terrain elevation (via TIN) from every point's Z
# lidR processes each tile in parallel and writes results to out_dir/normalized/
ctg_norm <- normalize_height(ctg_classified, algorithm = tin())

# Sanity check: load just the first tile (not the whole cloud) and verify
# ground points cluster near Z = 0
las_norm_check <- readLAS(list.files(file.path(out_dir, "normalized"),
                                      pattern = "\\.laz$", full.names = TRUE)[1])
hist(filter_ground(las_norm_check)$Z, main = "Ground point heights after normalization")

---

## 5. DSM from First Returns

The DSM represents the highest surface the lidar pulse hits — treetops,
rooftops, etc. Using only **first returns** ensures we capture the top
of the canopy rather than pulses that penetrated through gaps.

In [ ]:
# Load normalized tiles as a fresh catalog
ctg_norm <- readLAScatalog(list.files(file.path(out_dir, "normalized"),
                                       pattern = "\\.laz$", full.names = TRUE))

# Filter to first returns only at the catalog level — avoids loading all returns into memory
opt_filter(ctg_norm) <- "-keep_first"

# Return a single merged raster directly rather than one raster per tile
opt_output_files(ctg_norm) <- ""

dsm <- rasterize_canopy(
  ctg_norm,
  res       = 0.25,
  algorithm = p2r(subcircle = 0.1)  # point-to-raster; subcircle fills tiny gaps
  # Smoother alternative:
  # algorithm = dsmtin()
)

writeRaster(dsm, file.path(out_dir, "dsm_0.25m.tif"), overwrite = TRUE)
cat("DSM written to dsm_0.25m.tif\n")

---

## 6. CHM = DSM − DTM

Subtracting the DTM from the DSM removes terrain elevation, leaving only
**canopy height above ground**.

In [ ]:
# Align extents before subtracting (should already match, but just in case)
dtm_aligned <- resample(dtm, dsm, method = "bilinear")

chm <- dsm - dtm_aligned

# Clamp negative values to 0 (artefacts where DSM dips below DTM)
chm <- clamp(chm, lower = 0, values = TRUE)

writeRaster(chm, file.path(out_dir, "chm_0.25m.tif"), overwrite = TRUE)
cat("CHM written to chm_0.25m.tif\n")

---

## 7. Clip to AOI Boundary

In [ ]:
# Step 1: pop-up folder picker for the .gdb
gdb_path <- tcltk::tk_choose.dir(caption = "Select your .gdb folder")

# Step 2: list all layers and print them
layers <- st_layers(gdb_path)$name
cat("Layers found in geodatabase:\n")
for (i in seq_along(layers)) {
  cat(i, ":", layers[i], "\n")
}

# Step 3: type the number of the layer you want in the console when prompted
layer_index <- as.integer(readline(prompt = "Enter the number of the layer to use: "))
layer_name  <- layers[layer_index]
cat("Using layer:", layer_name, "\n")

# Step 4: read and reproject to match CHM
boundary <- st_read(gdb_path, layer = layer_name, quiet = TRUE)
boundary <- st_transform(boundary, crs = terra::crs(chm))

# Step 5: clip
chm_clipped <- mask(crop(chm, vect(boundary)), vect(boundary))

writeRaster(chm_clipped, file.path(out_dir, "chm_0.25m_clipped.tif"), overwrite = TRUE)
cat("Clipped CHM written to chm_0.25m_clipped.tif\n")

## 7b. Extract AOI Bounding Box for GEDI Data Pull

Reprojects the boundary to WGS84 (lat/long) and prints the bounding box
coordinates you can copy directly into your Python GEDI data pull script.

In [ ]:
# # Reproject boundary to WGS84 (EPSG:4326) for lat/long coordinates
# boundary_wgs84 <- st_transform(boundary, crs = 4326)

# # Get bounding box
# bb <- st_bbox(boundary_wgs84)

# cat("── AOI Bounding Box (WGS84) ──────────────────\n")
# cat(sprintf("  Min Longitude (West)  : %.6f\n", bb["xmin"]))
# cat(sprintf("  Max Longitude (East)  : %.6f\n", bb["xmax"]))
# cat(sprintf("  Min Latitude  (South) : %.6f\n", bb["ymin"]))
# cat(sprintf("  Max Latitude  (North) : %.6f\n", bb["ymax"]))
# cat("──────────────────────────────────────────────\n")
# cat("\nPython-ready (copy-paste):\n")
# cat(sprintf("LAT_MIN, LAT_MAX = %.3f, %.3f\nLON_MIN, LON_MAX = %.3f, %.3f\n",
#             bb["ymin"], bb["ymax"], bb["xmin"], bb["xmax"]))

---

## 8. Summary Statistics

Canopy height percentiles, matching the GEDI RH metric comparisons used
in Mishra & Singh (2026): RH95 ↔ P95, RH98 ↔ P98, etc.

In [ ]:
cat("\n── CHM summary (clipped) ──\n")
print(summary(values(chm_clipped), na.rm = TRUE))

chm_vals <- values(chm_clipped, na.rm = TRUE)
cat(sprintf("P95  = %.2f m\n", quantile(chm_vals, 0.95)))
cat(sprintf("P98  = %.2f m\n", quantile(chm_vals, 0.98)))
cat(sprintf("P99  = %.2f m\n", quantile(chm_vals, 0.99)))
cat(sprintf("P100 = %.2f m\n", max(chm_vals)))

---

## 9. Plot

In [ ]:
par(mfrow = c(1, 3))
plot(dtm,         main = "DTM (0.25 m)",         col = terrain.colors(100))
plot(dsm,         main = "DSM (0.25 m)",         col = terrain.colors(100))
plot(chm_clipped, main = "CHM clipped (0.25 m)", col = height.colors(100))

---

## References

Zhang, W., Qi, J., Wan, P., Wang, H., Xie, D., Wang, X., & Yan, G.
(2016). An easy-to-use airborne LiDAR data filtering method based on
cloth simulation. *Remote Sensing*, 8(6), 501.
<https://doi.org/10.3390/rs8060501>

Mishra, N. B., & Singh, P. B. (2026). Mountains of error: UAV LiDAR
benchmarking of GEDI and GEDI-derived global canopy height products in
steep Himalayan forests. *International Journal of Applied Earth
Observation and Geoinformation*, 147, 105203.

Yuwen Fu, Wang Li, Wensheng Duan, Jie Bai, Li Wang & Zheng Niu
(2025) Comparison of UAV-LiDAR-driven biomass estimation approaches in planted forests
with different management, International Journal of Digital Earth, 18:2, 2576910, DOI:
10.1080/17538947.2025.2576910